# Restricted Hartree--Fock (RHF) Method

In this tutorial we will make the first step -- code the restricted HF method.

## Import Section

The only two things we need to import is Psi4 and NumPy. 

In [1]:
import psi4
import numpy as np

Psi4 machinery is needed to specify a molecular system, to select a basis set, and to calculate molecular integrals in it. 
These are the most basic building blocks of any SCF program, and we are going to use them as black-box. We will cover them in detail in other tutorials, but for now we are just fine to use Psi4 software for these purposes.

NumPy is essential for any math-related manipulations. Namely, we will use it for matrix multiplication, matrix diagonalization, etc.

## Molecular Specification

Next, we need to specify a molecular system. It should be something simple, for starters, i.e. an atom.
Let us start with the He atom. It is a convinient "toy system" since there are only two electrons, they occupy a single orbital, and
the system is closed-shell.
For this system, we only need to specify the charge and multiplicity (0, 1).
In Psi4, it is made like this 

In [2]:
mol = psi4.geometry("""
0 1 
He
""")

(a variable's name "mol" will be used for any type of systems)

## Select a Basis Set

The choice of basis set is not a trivial task, one has to have an experience and some level of expertise to make the right choice. 
However, there are plenty of papers on the good practices in quantum chemistry, and DFT specifically, so we will be just following an 
advice from one of those papers. We are going to use basis sets of Ahlrichs and coworkers. Something not too big, just of the right size, like def2-TZVP. 

In [13]:
psi4.set_options({'basis': 'def2-TZVP'})

The set_options function allows tweaking a calculation setup in a lot of various ways. So far we only need to set a basis set for our program. This command basically instructs Psi4 "Go to a folder, where all the basis sets are stored, find a file with the name "def2-TZVP" and read the details about this basis set from there". What will happen if it does not find the file? We will check this in the next section.

## Mints and WFN Objects

Now, when we have specified the molecular system and the basis set that will represent it, it is time to compute the molecular integrals. 
Again, since we are not focusing on it right now, we will use Psi4 functions for that. In Psi4, this is done with Mints and WFN objects.
First, we use WFN to build a wavefunction of our molecule within the chosen basis set

In [14]:
wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option('basis'))

   => Loading Basis Set <=

    Name: DEF2-TZVP
    Role: ORBITAL
    Keyword: BASIS
    atoms 1 entry HE         line    27 file /opt/miniconda3/envs/p4env/share/psi4/basis/def2-tzvp.gbs 



Now we have recieved a message "=> Loading Basis Set <=" and the path to a chosen basis set on your machine. If you type something arbitrary like "'basis': 'def2-TZVPX'" above, you will simply get an error "BasisSetNotFound". But what if I entered a legit basis name and still got an error? Well, it means that this basis set is not in your library and you will need to add it there manually. Don't worry, in the basis set tutorial I explain all these things, it is not difficult at all. 

Now, let's compute molecular integrals for the kinetic $T$, electron-nuclear $V_{\text{ext}}$, electron-electron $V_{\text{ee}}$ (or, as they are often called, electron-repulsion integrals, ERI), and store them as variables

In [15]:
mints = psi4.core.MintsHelper(wfn.basisset())
T     = np.asarray(mints.ao_kinetic())        # Kinetic energy integrals
V     = np.asarray(mints.ao_potential())      # Electron-nuclear attraction integrals
EE    = np.asarray(mints.ao_eri())            # Electron-electron repulsion integrals